# Penggabungan Data Usaha Baru (6 Agustus 2026)

Notebook ini digunakan untuk menggabungkan seluruh file CSV data usaha baru yang terdapat di folder `data_usaha_baru/6Agustus2026`.
Semua kolom dibaca sebagai **string** (`dtype=str`) untuk memastikan angka `0` di depan maupun di belakang (seperti KBLI atau SLS) tidak hilang atau berubah menjadi float.

In [4]:
import os
import glob
import pandas as pd

# Path folder data usaha baru
folder_path = os.path.join("data_usaha_baru", "6Agustus2026")

# Mengambil semua file CSV di folder (mengecualikan file hasil penggabungan jika sudah ada)
all_csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
csv_files = [f for f in all_csv_files if not os.path.basename(f).startswith("data_usaha_baru_6Agustus2026")]

print(f"Ditemukan {len(csv_files)} file CSV yang akan digabungkan:")
for f in csv_files:
    print(f" - {os.path.basename(f)}")

# Membaca seluruh data CSV dengan tipe data String (dtype=str)
# agar angka nol di depan/belakang tidak terhapus atau berubah jadi float
df_list = [pd.read_csv(f, dtype=str) for f in csv_files]
df_merged = pd.concat(df_list, ignore_index=True)

# Menentukan path output dan menyimpan file hasil penggabungan
output_filename = "data_usaha_baru_6Agustus2026.csv"
output_path = os.path.join(folder_path, output_filename)
df_merged.to_csv(output_path, index=False)

print(f"\nPenggabungan selesai! Total {len(df_merged)} baris data berhasil disimpan ke {output_path}")

Ditemukan 4 file CSV yang akan digabungkan:
 - sqllab_untitled_query_12_20260806T114049.csv
 - sqllab_usaha_baru_blok_ii_10000_20260806T114513.csv
 - sqllab_usaha_baru_blok_ii_15000_20260806T114527.csv
 - sqllab_usaha_baru_blok_ii_sisa_20260806T114537.csv

Penggabungan selesai! Total 16748 baris data berhasil disimpan ke data_usaha_baru\6Agustus2026\data_usaha_baru_6Agustus2026.csv


In [5]:
# Menampilkan informasi ringkas dan 10 baris pertama data hasil penggabungan
print(df_merged.info())
df_merged.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16748 entries, 0 to 16747
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   assignment_id  16748 non-null  object
 1   nama_usaha     16748 non-null  object
 2   SLS            16748 non-null  object
 3   link_fasih     16748 non-null  object
 4   KBLI           16743 non-null  object
dtypes: object(5)
memory usage: 654.3+ KB
None


,assignment_id,nama_usaha,SLS,link_fasih,KBLI
0,0013195a-6858-435b-a2ae-bd240b15151b,USAHA TANAMAN PANGAN (EFRINE),71031000100003,https://fasih-sm.bps.go.id/app/assignment-deta...,101135
1,01836a91-2e61-4d76-adbb-56bbdc23f63c,INDUSTRI KOPRA(GUSTMUHS),71030620050002,https://fasih-sm.bps.go.id/app/assignment-deta...,10421
2,020d770c-043d-4063-808a-e99b7c6b999d,JUAL GORENGAN (JULIN),71030500060003,https://fasih-sm.bps.go.id/app/assignment-deta...,310794
3,020d770c-043d-4063-808a-e99b7c6b999d,Ojek motor (Anderson),71030500060003,https://fasih-sm.bps.go.id/app/assignment-deta...,149296
4,0218aa28-a2c6-4f86-bbf3-f63ff4ab0ab3,PERKEBUNAN PALA (AMELIA),71030500010003,https://fasih-sm.bps.go.id/app/assignment-deta...,101283
5,03314af4-d783-4688-87ec-6863a49547d1,PERKEBUNAN CENGKE,71030410060002,https://fasih-sm.bps.go.id/app/assignment-deta...,01282
6,03a29184-d14b-419f-b310-e73a5e65a97d,WARUNG SEMBAKO(ROS),71030500020001,https://fasih-sm.bps.go.id/app/assignment-deta...,347192
7,06fdb499-0486-4048-b173-7725762291a0,INDUSTRI PENGOLAHAN KOPRA (NELDI),71031100080002,https://fasih-sm.bps.go.id/app/assignment-deta...,10421
8,06fdb499-0486-4048-b173-7725762291a0,PERIKANAN TANGKAP (NELDI),71031100080002,https://fasih-sm.bps.go.id/app/assignment-deta...,03110
9,07064143-e280-414f-82eb-af7d2fd9f7bb,ANTRIX MART,71030600200001,https://fasih-sm.bps.go.id/app/assignment-deta...,247111


# Mapping Data Usaha Baru dengan Data Dashboard (Petugas & Kecamatan)

Section ini digunakan untuk melakukan mapping (join) antara data utama usaha baru (`data_usaha_baru_6Agustus2026.csv`) dengan data dashboard (`dashboard_scraped_data_evening.csv`) berdasarkan Kode SLS (14-digit).

Dari data dashboard, diambil informasi:
- **Nama Petugas** (`nama_pencacah` / `nama_pengawas`)
- **Jabatan** (`jabatan_pencacah` / `jabatan_pengawas`)
- **Nama Kecamatan** (`nama_kec`)

In [6]:
import os
import pandas as pd

# Path file input (mendukung eksekusi dari root folder maupun folder research)
path_main = os.path.join("data_usaha_baru", "6Agustus2026", "data_usaha_baru_6Agustus2026.csv")
path_dashboard = os.path.join("..", "dashboard_scraped_data_evening.csv")

if not os.path.exists(path_main):
    path_main = os.path.join("research", "data_usaha_baru", "6Agustus2026", "data_usaha_baru_6Agustus2026.csv")

if not os.path.exists(path_dashboard):
    path_dashboard = "dashboard_scraped_data_evening.csv"

print(f"Membaca data utama dari    : {path_main}")
print(f"Membaca data dashboard dari: {path_dashboard}")

# Membaca data dengan dtype=str agar leading zeros SLS/KBLI tidak hilang
df_main = pd.read_csv(path_main, dtype=str)
df_dash = pd.read_csv(path_dashboard, dtype=str)

# Menyamakan format Kode SLS (14 digit)
df_main['kode_sls_14'] = df_main['SLS'].astype(str).str.strip()
df_dash['kode_sls_14'] = df_dash['SLS Code'].astype(str).str.strip().str[:14]

# Agregasi data dashboard per SLS agar 1-to-1 mapping dengan data utama
dash_pencacah = df_dash[df_dash['Category'] == 'Pencacah'].drop_duplicates('kode_sls_14')[['kode_sls_14', 'nama_petugas', 'jabatan_petugas', 'nama_kec']].rename(
    columns={'nama_petugas': 'nama_pencacah', 'jabatan_petugas': 'jabatan_pencacah'}
)
dash_pengawas = df_dash[df_dash['Category'] == 'Pengawas'].drop_duplicates('kode_sls_14')[['kode_sls_14', 'nama_petugas', 'jabatan_petugas']].rename(
    columns={'nama_petugas': 'nama_pengawas', 'jabatan_petugas': 'jabatan_pengawas'}
)

dash_mapped = pd.merge(dash_pencacah, dash_pengawas, on='kode_sls_14', how='outer')

# Left join data utama dengan data dashboard yang sudah di-pivot
df_result = pd.merge(df_main, dash_mapped, on='kode_sls_14', how='left').drop(columns=['kode_sls_14'])

# Menentukan path output dan menyimpan file hasil mapping
output_folder = os.path.dirname(path_main)
output_path = os.path.join(output_folder, "data_usaha_baru_mapped_6Agustus2026.csv")
df_result.to_csv(output_path, index=False)

print(f"\nMapping selesai! Total {len(df_result)} baris data berhasil disimpan ke: {output_path}")

Membaca data utama dari    : data_usaha_baru\6Agustus2026\data_usaha_baru_6Agustus2026.csv
Membaca data dashboard dari: ..\dashboard_scraped_data_evening.csv

Mapping selesai! Total 16748 baris data berhasil disimpan ke: data_usaha_baru\6Agustus2026\data_usaha_baru_mapped_6Agustus2026.csv


In [7]:
# Menampilkan info ringkas dan 10 baris pertama hasil mapping
print(df_result.info())
df_result.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 16748 entries, 0 to 16747
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   assignment_id     16748 non-null  str  
 1   nama_usaha        16748 non-null  str  
 2   SLS               16748 non-null  str  
 3   link_fasih        16748 non-null  str  
 4   KBLI              16743 non-null  str  
 5   nama_pencacah     16748 non-null  str  
 6   jabatan_pencacah  16748 non-null  str  
 7   nama_kec          16748 non-null  str  
 8   nama_pengawas     16748 non-null  str  
 9   jabatan_pengawas  16748 non-null  str  
dtypes: str(10)
memory usage: 1.3 MB


,assignment_id,nama_usaha,SLS,link_fasih,KBLI,nama_pencacah,jabatan_pencacah,nama_kec,nama_pengawas,jabatan_pengawas
0,0013195a-6858-435b-a2ae-bd240b15151b,USAHA TANAMAN PANGAN (EFRINE),71031000100003,https://fasih-sm.bps.go.id/app/assignment-deta...,101135,Rubiani Mamuka,PPL,(100) TABUKAN UTARA,Derlan Malawere,PML
1,01836a91-2e61-4d76-adbb-56bbdc23f63c,INDUSTRI KOPRA(GUSTMUHS),71030620050002,https://fasih-sm.bps.go.id/app/assignment-deta...,10421,Mabelkhan Ladorang,PPL,(062) TABUKAN SELATAN TENGGARA,Ningsi Karendehi,PML
2,020d770c-043d-4063-808a-e99b7c6b999d,JUAL GORENGAN (JULIN),71030500060003,https://fasih-sm.bps.go.id/app/assignment-deta...,310794,Puteri Teratai Timuhingide,PPL,(050) TAMAKO,Fresly Jeica Manangkoda,PML
3,020d770c-043d-4063-808a-e99b7c6b999d,Ojek motor (Anderson),71030500060003,https://fasih-sm.bps.go.id/app/assignment-deta...,149296,Puteri Teratai Timuhingide,PPL,(050) TAMAKO,Fresly Jeica Manangkoda,PML
4,0218aa28-a2c6-4f86-bbf3-f63ff4ab0ab3,PERKEBUNAN PALA (AMELIA),71030500010003,https://fasih-sm.bps.go.id/app/assignment-deta...,101283,Billy Amstrong Tempolenehe,PPL,(050) TAMAKO,Fresly Jeica Manangkoda,PML
5,03314af4-d783-4688-87ec-6863a49547d1,PERKEBUNAN CENGKE,71030410060002,https://fasih-sm.bps.go.id/app/assignment-deta...,01282,Dince Lahaube,PPL,(041) TATOARENG,Kezia Maralending,PML
6,03a29184-d14b-419f-b310-e73a5e65a97d,WARUNG SEMBAKO(ROS),71030500020001,https://fasih-sm.bps.go.id/app/assignment-deta...,347192,Virchow Keynes Sahabat,PPL,(050) TAMAKO,Fresly Jeica Manangkoda,PML
7,06fdb499-0486-4048-b173-7725762291a0,INDUSTRI PENGOLAHAN KOPRA (NELDI),71031100080002,https://fasih-sm.bps.go.id/app/assignment-deta...,10421,Srirahayu Durumias,PPL,(110) KENDAHE,Hastuti Manabung,PML
8,06fdb499-0486-4048-b173-7725762291a0,PERIKANAN TANGKAP (NELDI),71031100080002,https://fasih-sm.bps.go.id/app/assignment-deta...,03110,Srirahayu Durumias,PPL,(110) KENDAHE,Hastuti Manabung,PML
9,07064143-e280-414f-82eb-af7d2fd9f7bb,ANTRIX MART,71030600200001,https://fasih-sm.bps.go.id/app/assignment-deta...,247111,Sry Ayu Permatasari Tamamile Manake,PPL,(060) TABUKAN SELATAN,Simson Petiunaung,PML


# Mapping Data Usaha Tidak Ditemukan dengan Data Dashboard (Petugas & Kecamatan)

Section ini digunakan untuk melakukan mapping (join) antara data utama usaha tidak ditemukan (`ga_ditemukan.csv`) dengan data dashboard (`dashboard_scraped_data_evening.csv`) berdasarkan Kode SLS (14-digit).

Dari data dashboard, diambil informasi:
- **Nama Petugas** (`nama_pencacah` / `nama_pengawas`)
- **Jabatan** (`jabatan_pencacah` / `jabatan_pengawas`)
- **Nama Kecamatan** (`nama_kec`)

In [8]:
import os
import pandas as pd

# Path file input (mendukung eksekusi dari root folder maupun folder research)
path_main = os.path.join("data_usaha_ga_ditemukan", "6Agustus2026", "ga_ditemukan.csv")
path_dashboard = os.path.join("..", "dashboard_scraped_data_evening.csv")

if not os.path.exists(path_main):
    path_main = os.path.join("research", "data_usaha_ga_ditemukan", "6Agustus2026", "ga_ditemukan.csv")

if not os.path.exists(path_dashboard):
    path_dashboard = "dashboard_scraped_data_evening.csv"

print(f"Membaca data utama dari    : {path_main}")
print(f"Membaca data dashboard dari: {path_dashboard}")

# Membaca data dengan dtype=str agar leading zeros SLS tidak hilang
df_main = pd.read_csv(path_main, dtype=str)
df_dash = pd.read_csv(path_dashboard, dtype=str)

# Menyamakan format Kode SLS (14 digit)
df_main['kode_sls_14'] = df_main['SLS'].astype(str).str.strip()
df_dash['kode_sls_14'] = df_dash['SLS Code'].astype(str).str.strip().str[:14]

# Agregasi data dashboard per SLS agar 1-to-1 mapping dengan data utama
dash_pencacah = df_dash[df_dash['Category'] == 'Pencacah'].drop_duplicates('kode_sls_14')[['kode_sls_14', 'nama_petugas', 'jabatan_petugas', 'nama_kec']].rename(
    columns={'nama_petugas': 'nama_pencacah', 'jabatan_petugas': 'jabatan_pencacah'}
)
dash_pengawas = df_dash[df_dash['Category'] == 'Pengawas'].drop_duplicates('kode_sls_14')[['kode_sls_14', 'nama_petugas', 'jabatan_petugas']].rename(
    columns={'nama_petugas': 'nama_pengawas', 'jabatan_petugas': 'jabatan_pengawas'}
)

dash_mapped = pd.merge(dash_pencacah, dash_pengawas, on='kode_sls_14', how='outer')

# Left join data utama dengan data dashboard yang sudah di-pivot
df_result_ga_ditemukan = pd.merge(df_main, dash_mapped, on='kode_sls_14', how='left').drop(columns=['kode_sls_14'])

# Menentukan path output dan menyimpan file hasil mapping
output_folder = os.path.dirname(path_main)
output_path = os.path.join(output_folder, "ga_ditemukan_mapped_6Agustus2026.csv")
df_result_ga_ditemukan.to_csv(output_path, index=False)

print(f"\nMapping selesai! Total {len(df_result_ga_ditemukan)} baris data berhasil disimpan ke: {output_path}")

Membaca data utama dari    : data_usaha_ga_ditemukan\6Agustus2026\ga_ditemukan.csv
Membaca data dashboard dari: ..\dashboard_scraped_data_evening.csv

Mapping selesai! Total 1574 baris data berhasil disimpan ke: data_usaha_ga_ditemukan\6Agustus2026\ga_ditemukan_mapped_6Agustus2026.csv


In [9]:
# Menampilkan info ringkas dan 10 baris pertama hasil mapping data usaha tidak ditemukan
print(df_result_ga_ditemukan.info())
df_result_ga_ditemukan.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 1574 entries, 0 to 1573
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   assignment_id         1574 non-null   str  
 1   SLS                   1574 non-null   str  
 2   nama_usaha_bang       1534 non-null   str  
 3   jenis_prelist         1574 non-null   str  
 4   ada_bang_usaha_label  1574 non-null   str  
 5   link_fasih            1574 non-null   str  
 6   nama_pencacah         1574 non-null   str  
 7   jabatan_pencacah      1574 non-null   str  
 8   nama_kec              1574 non-null   str  
 9   nama_pengawas         1574 non-null   str  
 10  jabatan_pengawas      1574 non-null   str  
dtypes: str(11)
memory usage: 135.4 KB


,assignment_id,SLS,nama_usaha_bang,jenis_prelist,ada_bang_usaha_label,link_fasih,nama_pencacah,jabatan_pencacah,nama_kec,nama_pengawas,jabatan_pengawas
0,0370b6ca-6331-47a8-9e9b-ec9f5392e9a5,71030500060002,SD GMIST BETHEL DAGHO (NONA HARTINA SJURIA ROM...,UMKM,0. Tidak Ditemukan,https://fasih-sm.bps.go.id/app/assignment-deta...,Puteri Teratai Timuhingide,PPL,(050) TAMAKO,Fresly Jeica Manangkoda,PML
1,068dd347-6d61-4ed7-a7e2-9420bf6bbcf3,71030900120010,CLERIZA MAKAWAEHE,OSS Perorangan,0. Tidak Ditemukan,https://fasih-sm.bps.go.id/app/assignment-deta...,Bayu Sutiyono Dujoh,PPL,(090) TAHUNA,George Muler Sasiang,PML
2,0cbc9178-d59e-49b8-8728-6e4f2cc10871,71030600150003,WARUNG SEMBAKO (RENI EFLIN TAMPATONDA),UMKM,0. Tidak Ditemukan,https://fasih-sm.bps.go.id/app/assignment-deta...,Meisye Fransiska Samau,PPL,(060) TABUKAN SELATAN,Simson Petiunaung,PML
3,109b2125-ae27-4f47-9344-b73ec9481681,71030910040003,KIOS MEGI (VEIBE GONI),UMKM,0. Tidak Ditemukan,https://fasih-sm.bps.go.id/app/assignment-deta...,Arianti Adahati,PPL,(091) TAHUNA TIMUR,Ivonne Lariunaung,PML
4,17f087b7-7448-434f-a7d8-1a97dfac1121,71030620010006,USAHA BENGKEL MOTOR IRVAN,UMKM,0. Tidak Ditemukan,https://fasih-sm.bps.go.id/app/assignment-deta...,Elan Chrisye Ansar,PPL,(062) TABUKAN SELATAN TENGGARA,Ningsi Karendehi,PML
5,364464b2-b5b6-48e1-bc9c-bc277f85c4a6,71031000050002,TOKO STENLY TANDRIS),UMKM,0. Tidak Ditemukan,https://fasih-sm.bps.go.id/app/assignment-deta...,Syamsia Makaminan,PPL,(100) TABUKAN UTARA,Yohana Sapar,PML
6,4340d7f5-1078-40bd-b27f-57536fbe83e2,71030900100003,NaN,UMKM,0. Tidak Ditemukan,https://fasih-sm.bps.go.id/app/assignment-deta...,Siti Nurhayati Igirisa,PPL,(090) TAHUNA,Yan Fredrik Wellian Padang,PML
7,44bb5bab-1ff8-445b-85a9-21992360d3aa,71031000090003,PENJUAL PULSA LISTRIK DAN HP (HENRA),UMKM,0. Tidak Ditemukan,https://fasih-sm.bps.go.id/app/assignment-deta...,Novianty Key,PPL,(100) TABUKAN UTARA,Novrilley Richard Mumu,PML
8,5fc13f44-90bf-4b80-ae9c-cba86ac4e69b,71030620010006,TAMBANG RAKYAT (JHON),UMKM,0. Tidak Ditemukan,https://fasih-sm.bps.go.id/app/assignment-deta...,Elan Chrisye Ansar,PPL,(062) TABUKAN SELATAN TENGGARA,Ningsi Karendehi,PML
9,66789f8f-4ac2-4ef9-a1bd-08189988f7a3,71031000280001,PARA PARA KOPRA SAMPING RUMAH JANABU MAKITULUNG,bangunan_lain,0. Tidak Ditemukan,https://fasih-sm.bps.go.id/app/assignment-deta...,Regina Mahoro,PPL,(100) TABUKAN UTARA,Yohana Sapar,PML


# Konversi Data Mapped (CSV) ke Excel (.xlsx)

Section ini digunakan untuk mengonversi file CSV hasil mapping (`data_usaha_baru_mapped_6Agustus2026.csv` dan `ga_ditemukan_mapped_6Agustus2026.csv`) ke format Excel (`.xlsx`).

Seluruh kolom dibaca dengan `dtype=str` untuk memastikan kolom-kolom identitas seperti Kode SLS (14-digit) dan KBLI (5-digit) tetap mempertahankan leading zeros dan tidak berubah menjadi format float/saintifik pada file Excel.

In [1]:
import os
import pandas as pd

# Path file CSV input (mendukung eksekusi dari root folder maupun folder research)
path_baru_csv = os.path.join("data_usaha_baru", "6Agustus2026", "data_usaha_baru_mapped_6Agustus2026.csv")
path_ga_ditemukan_csv = os.path.join("data_usaha_ga_ditemukan", "6Agustus2026", "ga_ditemukan_mapped_6Agustus2026.csv")

if not os.path.exists(path_baru_csv):
    path_baru_csv = os.path.join("research", "data_usaha_baru", "6Agustus2026", "data_usaha_baru_mapped_6Agustus2026.csv")

if not os.path.exists(path_ga_ditemukan_csv):
    path_ga_ditemukan_csv = os.path.join("research", "data_usaha_ga_ditemukan", "6Agustus2026", "ga_ditemukan_mapped_6Agustus2026.csv")

# Path file Excel output
path_baru_xlsx = path_baru_csv.replace(".csv", ".xlsx")
path_ga_ditemukan_xlsx = path_ga_ditemukan_csv.replace(".csv", ".xlsx")

print("Mengonversi CSV ke Excel...")

# 1. Konversi Data Usaha Baru Mapped
df_baru = pd.read_csv(path_baru_csv, dtype=str)
try:
    df_baru.to_excel(path_baru_xlsx, index=False)
    print(f"✓ Data Usaha Baru Mapped berhasil disimpan ke Excel : {path_baru_xlsx} ({len(df_baru)} baris, {len(df_baru.columns)} kolom)")
except PermissionError:
    print(f"⚠️ GAGAL MENULIS: File '{path_baru_xlsx}' sedang dibuka oleh Microsoft Excel. Tutup file tersebut terlebih dahulu lalu jalankan ulang cell ini.")

# 2. Konversi Data Usaha Tidak Ditemukan Mapped
df_ga_ditemukan = pd.read_csv(path_ga_ditemukan_csv, dtype=str)
try:
    df_ga_ditemukan.to_excel(path_ga_ditemukan_xlsx, index=False)
    print(f"✓ Data Usaha Tidak Ditemukan Mapped berhasil disimpan ke Excel: {path_ga_ditemukan_xlsx} ({len(df_ga_ditemukan)} baris, {len(df_ga_ditemukan.columns)} kolom)")
except PermissionError:
    print(f"⚠️ GAGAL MENULIS: File '{path_ga_ditemukan_xlsx}' sedang dibuka oleh Microsoft Excel. Tutup file tersebut terlebih dahulu lalu jalankan ulang cell ini.")


Mengonversi CSV ke Excel...
✓ Data Usaha Baru Mapped berhasil disimpan ke Excel : data_usaha_baru\6Agustus2026\data_usaha_baru_mapped_6Agustus2026.xlsx (16748 baris, 10 kolom)
✓ Data Usaha Tidak Ditemukan Mapped berhasil disimpan ke Excel: data_usaha_ga_ditemukan\6Agustus2026\ga_ditemukan_mapped_6Agustus2026.xlsx (1574 baris, 11 kolom)


In [3]:
# Verifikasi struktur kolom dan tipe data Excel yang telah dibuat
print("=== VERIFIKASI KOLOM DATA USAHA BARU MAPPED ===")
print(f"Jumlah Kolom: {len(df_baru.columns)}")
print("Daftar Kolom:", df_baru.columns.tolist())

print("\n=== VERIFIKASI KOLOM DATA USAHA TIDAK DITEMUKAN MAPPED ===")
print(f"Jumlah Kolom: {len(df_ga_ditemukan.columns)}")
print("Daftar Kolom:", df_ga_ditemukan.columns.tolist())


=== VERIFIKASI KOLOM DATA USAHA BARU MAPPED ===
Jumlah Kolom: 10
Daftar Kolom: ['assignment_id', 'nama_usaha', 'SLS', 'link_fasih', 'KBLI', 'nama_pencacah', 'jabatan_pencacah', 'nama_kec', 'nama_pengawas', 'jabatan_pengawas']

=== VERIFIKASI KOLOM DATA USAHA TIDAK DITEMUKAN MAPPED ===
Jumlah Kolom: 11
Daftar Kolom: ['assignment_id', 'SLS', 'nama_usaha_bang', 'jenis_prelist', 'ada_bang_usaha_label', 'link_fasih', 'nama_pencacah', 'jabatan_pencacah', 'nama_kec', 'nama_pengawas', 'jabatan_pengawas']
